# Day 02: The Unscented Kalman Filter (UKF) - Nonlinear Manifolds & Sigma-Point Filtering
**State Estimation and Localization for Self-Driving Cars**

### 🎯 Learning Objectives:
1. Understand the fundamental limitation of Taylor series linearization in highly curved manifolds.
2. Master the **Standard Non-Scaled Unscented Transform (UT)** and parameter $\kappa$.
3. Implement the generic multi-dimensional `UnscentedKalmanFilter` class via robust Cholesky matrix decomposition and circular angle mean unwrapping.
4. Benchmark UKF vs. EKF on high-rate turning maneuvers with severe polar bearing nonlinearities.
5. Visualize covariance confidence bounds and residual convergence via interactive Plotly dashboards.

---
## 1. Environment Setup

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
print('Environment ready: NumPy and Plotly loaded.')

---
## 2. Mathematical Foundation: The Unscented Kalman Filter (UKF)

We can easily use the Unscented Transform in our Kalman Filtering framework with nonlinear models:

$$\begin{aligned}
\textbf{Nonlinear motion model:} \quad &\mathbf{x}_k = \mathbf{f}_{k-1}(\mathbf{x}_{k-1},\mathbf{u}_{k-1},\mathbf{w}_{k-1}), \quad \mathbf{w}_k \sim \mathcal{N}(\mathbf{0},\mathbf{Q}_k) \\[6pt]
\textbf{Nonlinear measurement model:} \quad &\mathbf{y}_k = \mathbf{h}_k(\mathbf{x}_k,\mathbf{v}_k), \quad \mathbf{v}_k \sim \mathcal{N}(\mathbf{0},\mathbf{R}_k)
\end{aligned}$$

Instead of approximating system equations by linearizing, we calculate sigma points and use the Unscented Transform to propagate the probability density functions directly!

### 1️⃣ Prediction Step
To propagate the state from time $(k-1)$ to time $k$, apply the Unscented Transform using the current estimate $(\mathbf{\hat{x}}_{k-1}, \mathbf{\hat{P}}_{k-1})$:

1. **Compute $2N+1$ Sigma Points:**
   $$\mathbf{\hat{L}}_{k-1}\mathbf{\hat{L}}_{k-1}^T = \mathbf{\hat{P}}_{k-1}$$
   $$\mathbf{\hat{x}}_{k-1}^{(0)} = \mathbf{\hat{x}}_{k-1}$$
   $$\mathbf{\hat{x}}_{k-1}^{(i)} = \mathbf{\hat{x}}_{k-1} + \sqrt{N+\kappa}\,\operatorname{col}_i(\mathbf{\hat{L}}_{k-1}) \quad (i=1,\dots,N)$$
   $$\mathbf{\hat{x}}_{k-1}^{(i+N)} = \mathbf{\hat{x}}_{k-1} - \sqrt{N+\kappa}\,\operatorname{col}_i(\mathbf{\hat{L}}_{k-1}) \quad (i=1,\dots,N)$$

2. **Propagate Sigma Points through Motion Model:**
   $$\mathbf{\check{x}}_k^{(i)} = \mathbf{f}_{k-1}(\mathbf{\hat{x}}_{k-1}^{(i)}, \mathbf{u}_{k-1}, \mathbf{0}) \quad (i=0,\dots,2N)$$

3. **Compute Predicted Mean and Covariance:**
   With non-scaled weights $\alpha^{(i)}$:
   $$\alpha^{(0)} = \frac{\kappa}{N+\kappa}, \quad \alpha^{(i)} = \frac{1}{2(N+\kappa)} \quad (i=1,\dots,2N)$$
   $$\mathbf{\check{x}}_k = \sum_{i=0}^{2N} \alpha^{(i)}\mathbf{\check{x}}_k^{(i)}, \quad \mathbf{\check{P}}_k = \sum_{i=0}^{2N} \alpha^{(i)}(\mathbf{\check{x}}_k^{(i)} - \mathbf{\check{x}}_k)(\mathbf{\check{x}}_k^{(i)} - \mathbf{\check{x}}_k)^T + \mathbf{Q}_{k-1}$$

### 2️⃣ Correction / Measurement Update Step

1. **Propagate Sigma Points through Measurement Model:**
   $$\mathbf{\hat{y}}_k^{(i)} = \mathbf{h}_k(\mathbf{\check{x}}_k^{(i)}, \mathbf{0}) \quad (i=0,\dots,2N)$$

2. **Compute Predicted Measurement and Innovation Covariance:**
   $$\mathbf{\hat{y}}_k = \sum_{i=0}^{2N} \alpha^{(i)}\mathbf{\hat{y}}_k^{(i)}, \quad \mathbf{P}_y = \sum_{i=0}^{2N} \alpha^{(i)}(\mathbf{\hat{y}}_k^{(i)} - \mathbf{\hat{y}}_k)(\mathbf{\hat{y}}_k^{(i)} - \mathbf{\hat{y}}_k)^T + \mathbf{R}_k$$

3. **Compute Cross-Covariance:**
   $$\mathbf{P}_{xy} = \sum_{i=0}^{2N} \alpha^{(i)}(\mathbf{\check{x}}_k^{(i)} - \mathbf{\check{x}}_k)(\mathbf{\hat{y}}_k^{(i)} - \mathbf{\hat{y}}_k)^T$$

4. **Optimal Kalman Gain & Posterior State Update:**
   $$\mathbf{K}_k = \mathbf{P}_{xy}\mathbf{P}_y^{-1}, \quad \mathbf{\hat{x}}_k = \mathbf{\check{x}}_k + \mathbf{K}_k(\mathbf{y}_k - \mathbf{\hat{y}}_k), \quad \mathbf{\hat{P}}_k = \mathbf{\check{P}}_k - \mathbf{K}_k \mathbf{P}_y \mathbf{K}_k^T$$

---
## 3. Generic `UnscentedKalmanFilter` Class Implementation

In [ ]:
def wrap_angle(angle):
    """Wraps angle to [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class UnscentedKalmanFilter:
    """Generic Multi-Dimensional Unscented Kalman Filter (UKF) with Standard Non-Scaled UT.
    
    Completely derivative-free state estimation supporting arbitrary dimension N, nonlinear
    transition functions f(x, u), observation models h(x), and robust angle unwrapping.
    """
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray, kappa: float = 0.0):
        self.x = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.N = self.x.shape[0]
        self.L = self.N
        self.kappa = float(kappa)
        
        self.gamma = np.sqrt(self.N + self.kappa)
        self.num_sigmas = 2 * self.N + 1
        
        # Standard Non-Scaled Weights alpha^(i)
        denom = self.N + self.kappa
        self.alpha_weights = np.full(self.num_sigmas, 1.0 / (2.0 * denom))
        self.alpha_weights[0] = self.kappa / denom
        self.Wm = self.alpha_weights
        self.Wc = self.alpha_weights
            
        self.latest_innovation = None
        self.latest_innovation_cov = None
        self.latest_gain = None
            
    def generate_sigma_points(self, x_mean: np.ndarray, P_cov: np.ndarray, angle_indices: list = None):
        """Generates 2N+1 sigma points via robust Cholesky decomposition L = chol(P)."""
        P_sym = 0.5 * (P_cov + P_cov.T)
        try:
            L_mat = np.linalg.cholesky(P_sym)
        except np.linalg.LinAlgError:
            eigvals, eigvecs = np.linalg.eigh(P_sym)
            eigvals = np.maximum(eigvals, 1e-9)
            P_sym = eigvecs @ np.diag(eigvals) @ eigvecs.T
            L_mat = np.linalg.cholesky(P_sym)
            
        sigmas = np.zeros((self.N, self.num_sigmas))
        sigmas[:, 0] = x_mean.flatten()
        for i in range(self.N):
            col = L_mat[:, i]
            sigmas[:, i + 1] = x_mean.flatten() + self.gamma * col
            sigmas[:, i + 1 + self.N] = x_mean.flatten() - self.gamma * col
        if angle_indices is not None:
            for idx in angle_indices:
                sigmas[idx, :] = wrap_angle(sigmas[idx, :])
        return sigmas
        
    def predict(self, f_func, Q: np.ndarray, u: np.ndarray = None, angle_indices: list = None):
        """Executes the UKF Sigma-Point State and Covariance Prediction Step."""
        sigmas = self.generate_sigma_points(self.x, self.P, angle_indices=angle_indices)
        sigmas_pred = np.zeros_like(sigmas)
        for i in range(self.num_sigmas):
            sp = sigmas[:, i:i+1]
            if u is not None:
                sigmas_pred[:, i:i+1] = f_func(sp, u).reshape(-1, 1)
            else:
                sigmas_pred[:, i:i+1] = f_func(sp).reshape(-1, 1)
            
        # Recombine mean with circular angle unwrapping
        x_pred = np.zeros((self.N, 1))
        for row in range(self.N):
            if angle_indices is not None and row in angle_indices:
                ref = sigmas_pred[row, 0]
                unwrapped = ref + wrap_angle(sigmas_pred[row, :] - ref)
                x_pred[row, 0] = wrap_angle(np.sum(self.Wm * unwrapped))
            else:
                x_pred[row, 0] = np.sum(self.Wm * sigmas_pred[row, :])
                
        P_pred = np.zeros((self.N, self.N))
        for i in range(self.num_sigmas):
            diff = sigmas_pred[:, i:i+1] - x_pred
            if angle_indices is not None:
                for idx in angle_indices:
                    diff[idx, 0] = wrap_angle(diff[idx, 0])
            P_pred += self.Wc[i] * (diff @ diff.T)
        P_pred += Q
        
        self.x = x_pred
        self.P = 0.5 * (P_pred + P_pred.T)
        return self.x.copy(), self.P.copy()
        
    def update(self, y: np.ndarray, h_func, R: np.ndarray, 
               meas_angle_indices: list = None, state_angle_indices: list = None):
        """Executes the UKF Measurement Correction Step."""
        y_vec = np.asarray(y, dtype=np.float64).reshape(-1, 1)
        m = y_vec.shape[0]
        sigmas = self.generate_sigma_points(self.x, self.P, angle_indices=state_angle_indices)
        gamma_meas = np.zeros((m, self.num_sigmas))
        for i in range(self.num_sigmas):
            gamma_meas[:, i:i+1] = h_func(sigmas[:, i:i+1]).reshape(-1, 1)
            
        # Recombine measurement mean with circular angle unwrapping
        y_pred = np.zeros((m, 1))
        for row in range(m):
            if meas_angle_indices is not None and row in meas_angle_indices:
                ref = gamma_meas[row, 0]
                unwrapped = ref + wrap_angle(gamma_meas[row, :] - ref)
                y_pred[row, 0] = wrap_angle(np.sum(self.Wm * unwrapped))
            else:
                y_pred[row, 0] = np.sum(self.Wm * gamma_meas[row, :])
                
        Py = np.zeros((m, m))
        Pxy = np.zeros((self.N, m))
        for i in range(self.num_sigmas):
            dy = gamma_meas[:, i:i+1] - y_pred
            if meas_angle_indices is not None:
                for idx in meas_angle_indices:
                    dy[idx, 0] = wrap_angle(dy[idx, 0])
            dx = sigmas[:, i:i+1] - self.x
            if state_angle_indices is not None:
                for idx in state_angle_indices:
                    dx[idx, 0] = wrap_angle(dx[idx, 0])
            Py += self.Wc[i] * (dy @ dy.T)
            Pxy += self.Wc[i] * (dx @ dy.T)
        Py += R
        
        K = Pxy @ np.linalg.inv(Py)
        residual = y_vec - y_pred
        if meas_angle_indices is not None:
            for idx in meas_angle_indices:
                residual[idx, 0] = wrap_angle(residual[idx, 0])
                
        self.x = self.x + K @ residual
        if state_angle_indices is not None:
            for idx in state_angle_indices:
                self.x[idx, 0] = wrap_angle(self.x[idx, 0])
                
        self.P = self.P - K @ Py @ K.T
        self.P = 0.5 * (self.P + self.P.T)
        
        self.latest_innovation = residual
        self.latest_innovation_cov = Py
        self.latest_gain = K
        
        return self.x.copy(), self.P.copy()

print('Generic UnscentedKalmanFilter class compiled successfully.')

### 🧪 Unit Test: Generic UnscentedKalmanFilter Sanity Check (Sigma Points, Propagation & Update)

In [ ]:
# Sanity verification of generic UnscentedKalmanFilter class
x0_t = np.array([[10.0], [2.0]])
P0_t = np.eye(2) * 4.0
ukf_t = UnscentedKalmanFilter(x0=x0_t, P0=P0_t, kappa=1.0)

# 1. Test Non-Scaled Weights alpha^(i)
assert np.isclose(np.sum(ukf_t.alpha_weights), 1.0), 'Sum of alpha weights must equal 1.0'
assert np.isclose(np.sum(ukf_t.Wm), 1.0), 'Sum of Wm must equal 1.0'
assert ukf_t.num_sigmas == 5, f'Expected 5 sigma points for N=2, got {ukf_t.num_sigmas}'
assert np.isclose(ukf_t.alpha_weights[0], 1.0 / 3.0), f'Expected alpha[0]=1/3 for N=2, kappa=1, got {ukf_t.alpha_weights[0]}'
assert np.isclose(ukf_t.alpha_weights[1], 1.0 / 6.0), f'Expected alpha[1]=1/6 for N=2, kappa=1, got {ukf_t.alpha_weights[1]}'

# 2. Test Sigma Point Generation
sigmas = ukf_t.generate_sigma_points(x0_t, P0_t)
assert sigmas.shape == (2, 5), f'Expected shape (2, 5), got {sigmas.shape}'
recon_mean = np.sum(ukf_t.Wm * sigmas, axis=1, keepdims=True)
assert np.allclose(recon_mean, x0_t, atol=1e-5), f'Reconstructed mean {recon_mean.T} != {x0_t.T}'

# 3. Test Prediction Step
dt_t = 0.1
f_t = lambda x, u=None: np.array([[x[0, 0] + dt_t * x[1, 0]], [x[1, 0]]])
Q_t = np.diag([0.01, 0.05])
x_p, P_p = ukf_t.predict(f_func=f_t, Q=Q_t)
assert x_p.shape == (2, 1), f'Expected shape (2, 1), got {x_p.shape}'
assert np.isclose(x_p[0, 0], 10.2, atol=1e-4), f'Expected pos 10.2, got {x_p[0, 0]}'
assert np.isclose(x_p[1, 0], 2.0, atol=1e-4), f'Expected vel 2.0, got {x_p[1, 0]}'
assert P_p.shape == (2, 2), f'Expected shape (2, 2), got {P_p.shape}'
assert P_p[0, 0] > 4.0, 'Covariance must expand after prediction'

# 4. Test Measurement Update Step
h_t = lambda x: np.array([[x[0, 0]]])
R_t = np.array([[0.25]])
x_u, P_u = ukf_t.update(y=np.array([[10.5]]), h_func=h_t, R=R_t)
assert x_u.shape == (2, 1), f'Expected shape (2, 1), got {x_u.shape}'
assert P_u.shape == (2, 2), f'Expected shape (2, 2), got {P_u.shape}'
assert P_u[0, 0] < P_p[0, 0], 'Covariance must decrease after update'
assert np.all(np.linalg.eigvals(P_u) > 0), 'P must remain strictly positive definite!'
print('✅ UnscentedKalmanFilter generic class unit tests passed!')

---
## 4. Analytical Linearization (EKF) vs. Derivative-Free Sampling (UKF)

To benchmark UKF against EKF under aggressive nonlinear vehicle maneuvers:

### 📐 The Classical EKF Partial Derivatives:
Recall that the baseline EKF relies on locally evaluating the Jacobian matrices $\mathbf{F} = \frac{\partial \mathbf{f}}{\partial \mathbf{x}}$ and $\mathbf{H} = \frac{\partial \mathbf{h}}{\partial \mathbf{x}}$:
$$\mathbf{F} = \begin{bmatrix} 1 & 0 & \cos(\theta)\Delta t & -v \sin(\theta)\Delta t \\ 0 & 1 & \sin(\theta)\Delta t & v \cos(\theta)\Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}, \quad
\mathbf{H} = \begin{bmatrix} \frac{p_x}{r} & \frac{p_y}{r} & 0 & 0 \\[6pt] -\frac{p_y}{r^2} & \frac{p_x}{r^2} & 0 & 0 \end{bmatrix}, \quad r = \sqrt{p_x^2 + p_y^2}$$

**Why EKF degrades under severe angular rates and bearing noise:**
- EKF truncates the Taylor expansion at $\mathcal{O}(\Delta \mathbf{x}^1)$, ignoring curvature terms $\frac{1}{2}\Delta\mathbf{x}^T \nabla^2 \mathbf{f}\,\Delta\mathbf{x}$. When $\sigma_\phi$ is large or yaw rates are high, the prior Gaussian is severely distorted into a non-Gaussian banana-shaped manifold.
- In contrast, the **UKF** propagates $2N+1 = 9$ deterministic sigma points through the exact nonlinear equations $\mathbf{f}(\cdot)$ and $\mathbf{h}(\cdot)$, capturing posterior mean and covariance with 2nd-order Taylor accuracy without evaluating any matrix of partial derivatives.

---
## 5. Benchmark Challenge: EKF vs. UKF on Aggressive Nonlinear Maneuver

In [ ]:
dt = 0.1
T_total = 40.0
N_steps = int(T_total / dt)
time = np.linspace(0, T_total, N_steps)

x_true = np.array([10.0, 5.0, 12.0, 0.0]).reshape(4, 1)
x_true_all = np.zeros((4, N_steps))

Q = np.diag([0.05**2, 0.05**2, 0.1**2, 0.03**2])
R = np.diag([0.8**2, np.deg2rad(3.0)**2])

def motion_model(x, u):
    px, py, v, theta = x.flatten()
    a, omega = u.flatten()
    px_next = px + v * np.cos(theta) * dt
    py_next = py + v * np.sin(theta) * dt
    v_next = v + a * dt
    theta_next = wrap_angle(theta + omega * dt)
    return np.array([px_next, py_next, v_next, theta_next]).reshape(4, 1)

def measurement_model(x):
    px, py = x[0, 0], x[1, 0]
    r = np.sqrt(px**2 + py**2)
    phi = np.arctan2(py, px)
    return np.array([r, phi]).reshape(2, 1)

def get_F_jacobian(x):
    px, py, v, theta = x.flatten()
    return np.array([
        [1.0, 0.0, np.cos(theta) * dt, -v * np.sin(theta) * dt],
        [0.0, 1.0, np.sin(theta) * dt,  v * np.cos(theta) * dt],
        [0.0, 0.0, 1.0,                 0.0],
        [0.0, 0.0, 0.0,                 1.0]
    ])

def get_H_jacobian(x):
    px, py = x[0, 0], x[1, 0]
    r2 = max(px**2 + py**2, 1e-6)
    r = np.sqrt(r2)
    return np.array([
        [px / r,       py / r,       0.0, 0.0],
        [-py / r2,     px / r2,      0.0, 0.0]
    ])

measurements = []
for k in range(N_steps):
    t = time[k]
    a = 0.5 * np.cos(0.2 * t)
    omega = 0.25 * np.sin(0.15 * t)
    u = np.array([a, omega]).reshape(2, 1)
    w = np.random.multivariate_normal(np.zeros(4), Q).reshape(4, 1)
    x_true = motion_model(x_true, u) + w
    x_true[3, 0] = wrap_angle(x_true[3, 0])
    x_true_all[:, k] = x_true.flatten()
    v_noise = np.random.multivariate_normal(np.zeros(2), R).reshape(2, 1)
    y = measurement_model(x_true) + v_noise
    y[1, 0] = wrap_angle(y[1, 0])
    measurements.append((u, y))

x0_init = np.array([8.0, 3.0, 10.0, np.deg2rad(10.0)]).reshape(4, 1)
P0_init = np.diag([5.0**2, 5.0**2, 5.0**2, np.deg2rad(20.0)**2])

x_ekf = x0_init.copy()
P_ekf = P0_init.copy()
x_ekf_all = np.zeros((4, N_steps))

ukf = UnscentedKalmanFilter(x0_init, P0_init, kappa=0.0)
x_ukf_all = np.zeros((4, N_steps))

for k in range(N_steps):
    u, y = measurements[k]
    # EKF Execution
    F = get_F_jacobian(x_ekf)
    x_ekf = motion_model(x_ekf, u)
    P_ekf = F @ P_ekf @ F.T + Q
    H = get_H_jacobian(x_ekf)
    y_pred_ekf = measurement_model(x_ekf)
    res_ekf = y - y_pred_ekf
    res_ekf[1, 0] = wrap_angle(res_ekf[1, 0])
    S_ekf = H @ P_ekf @ H.T + R
    K_ekf = P_ekf @ H.T @ np.linalg.inv(S_ekf)
    x_ekf = x_ekf + K_ekf @ res_ekf
    x_ekf[3, 0] = wrap_angle(x_ekf[3, 0])
    P_ekf = (np.eye(4) - K_ekf @ H) @ P_ekf
    x_ekf_all[:, k] = x_ekf.flatten()

    # Generic UKF Execution
    ukf.predict(motion_model, Q, u=u, angle_indices=[3])
    ukf.update(y, measurement_model, R, meas_angle_indices=[1], state_angle_indices=[3])
    x_ukf_all[:, k] = ukf.x.flatten()

rmse_ekf_pos = np.sqrt(np.mean((x_true_all[0, :] - x_ekf_all[0, :])**2 + (x_true_all[1, :] - x_ekf_all[1, :])**2))
rmse_ukf_pos = np.sqrt(np.mean((x_true_all[0, :] - x_ukf_all[0, :])**2 + (x_true_all[1, :] - x_ukf_all[1, :])**2))

print(f'Position RMSE -> EKF: {rmse_ekf_pos:.3f} m | UKF: {rmse_ukf_pos:.3f} m')

---
## 6. Comparative Visualizations: UKF vs. EKF via Plotly

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'Trajectory Tracking: EKF vs. UKF (EKF: {rmse_ekf_pos:.2f}m, UKF: {rmse_ukf_pos:.2f}m)',
        'Position Error Convergence Over Time'
    )
)

fig.add_trace(go.Scatter(x=x_true_all[0, :], y=x_true_all[1, :], mode='lines', name='Ground Truth', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_ekf_all[0, :], y=x_ekf_all[1, :], mode='lines', name=f'EKF (RMSE: {rmse_ekf_pos:.2f}m)', line=dict(color='red', dash='dash', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_ukf_all[0, :], y=x_ukf_all[1, :], mode='lines', name=f'UKF (RMSE: {rmse_ukf_pos:.2f}m)', line=dict(color='blue', width=2)), row=1, col=1)

err_ekf = np.sqrt((x_true_all[0, :] - x_ekf_all[0, :])**2 + (x_true_all[1, :] - x_ekf_all[1, :])**2)
err_ukf = np.sqrt((x_true_all[0, :] - x_ukf_all[0, :])**2 + (x_true_all[1, :] - x_ukf_all[1, :])**2)
fig.add_trace(go.Scatter(x=time, y=err_ekf, mode='lines', name='EKF Error [m]', line=dict(color='red', dash='dash', width=1.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=time, y=err_ukf, mode='lines', name='UKF Error [m]', line=dict(color='blue', width=1.5)), row=1, col=2)

fig.update_layout(title_text='Day 2 UKF vs. EKF Benchmark: Nonlinear Manifold Tracking', template='plotly_white', height=500, width=1100)
fig.show()

---
## 7. Summary
The UKF avoids Jacobians entirely, capturing true probability moments up to 3rd order with significantly greater resilience to severe non-linearities.